In [4]:
import numpy as np

sites = [
    "GAGGTAAAC",
    "TCCGTAAGC",
    "CAGGTTGGA",
    "ACAGTCAGC",
    "TAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTCGAT",
    "CAGGTCAGC",
    "CAGGTCAGC",
    "CAGGTTGGC"
]

NUCLEOTIDES = ['A', 'C', 'G', 'T']
N = len(sites)
L = len(sites[0])

def build_pfm(sites):
    """
    Строит матрицу частот нуклеотидов по позициям.
    Возвращает numpy array размера (4, L), где строки: A, C, G, T
    """
    pfm = np.zeros((4, L), dtype=int)

    for i in range(L):
        for j, site in enumerate(sites):
            nucleotide = site[i]
            idx = NUCLEOTIDES.index(nucleotide)
            pfm[idx, i] += 1

    return pfm

def pfm_to_ppm(pfm, alpha=0.1):
    """
    Преобразует PFM в PPM с добавлением псевдосчётов.
    """
    pfm_pseudo = pfm + alpha

    column_sums = np.sum(pfm_pseudo, axis=0)
    ppm = pfm_pseudo / column_sums

    return ppm

def ppm_to_pwm(ppm, background):
    """
    Преобразует PPM в PWM.
    background: dict с фоновыми частотами {'A': 0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}
    """
    bg_array = np.array([background['A'], background['C'], background['G'], background['T']])
    bg_array = bg_array.reshape(4, 1)

    epsilon = 1e-10
    pwm = np.log2((ppm + epsilon) / (bg_array + epsilon))

    return pwm

pfm = build_pfm(sites)
print("\n1. PFM (Position Frequency Matrix):")
print("      ", end="")
for i in range(L):
    print(f"Поз.{i+1:2d}", end=" ")
print()
for nuc_idx, nuc in enumerate(NUCLEOTIDES):
    print(f"  {nuc}: ", end="")
    for i in range(L):
        print(f"{pfm[nuc_idx, i]:6d}", end=" ")
    print()

ppm = pfm_to_ppm(pfm, alpha=0.1)
print("\n2. PPM с псевдосчётом α=0.1 (округлено до 3 знаков):")
print("      ", end="")
for i in range(L):
    print(f"Поз.{i+1:2d}", end=" ")
print()
for nuc_idx, nuc in enumerate(NUCLEOTIDES):
    print(f"  {nuc}: ", end="")
    for i in range(L):
        print(f"{ppm[nuc_idx, i]:6.3f}", end=" ")
    print()

print("\n   Проверка сумм по столбцам:")
column_sums = np.sum(ppm, axis=0)
for i, s in enumerate(column_sums):
    print(f"     Поз.{i+1}: {s:.6f}")

background = {'A': 0.295, 'C': 0.205, 'G': 0.205, 'T': 0.295}
pwm = ppm_to_pwm(ppm, background)

print("\n3. PWM (веса в битах, log2(P/фон), округлено до 3 знаков):")
print("      ", end="")
for i in range(L):
    print(f"Поз.{i+1:2d}", end=" ")
print()
for nuc_idx, nuc in enumerate(NUCLEOTIDES):
    print(f"  {nuc}: ", end="")
    for i in range(L):
        print(f"{pwm[nuc_idx, i]:6.3f}", end=" ")
    print()

max_score = 0
min_score = 0
max_sequence = [""] * L
min_sequence = [""] * L

for i in range(L):
    max_idx = np.argmax(pwm[:, i])
    min_idx = np.argmin(pwm[:, i])

    max_score += pwm[max_idx, i]
    min_score += pwm[min_idx, i]

    max_sequence[i] = NUCLEOTIDES[max_idx]
    min_sequence[i] = NUCLEOTIDES[min_idx]

print(f"\n4. Теоретические экстремумы:")
print(f"   Максимальный скор: {max_score:.4f}")
print(f"   Последовательность с макс. скором: {''.join(max_sequence)}")
print(f"   Минимальный скор: {min_score:.4f}")
print(f"   Последовательность с мин. скором: {''.join(min_sequence)}")

print("\n   Проверка: скоры исходных последовательностей:")
scores = []
for site in sites:
    score = 0
    for pos, nuc in enumerate(site):
        nuc_idx = NUCLEOTIDES.index(nuc)
        score += pwm[nuc_idx, pos]
    scores.append(score)
    print(f"     {site}: {score:.4f}")

print(f"\n   Средний скор по выборке: {np.mean(scores):.4f}")
print(f"   Минимальный скор в выборке: {np.min(scores):.4f}")
print(f"   Максимальный скор в выборке: {np.max(scores):.4f}")


1. PFM (Position Frequency Matrix):
      Поз. 1 Поз. 2 Поз. 3 Поз. 4 Поз. 5 Поз. 6 Поз. 7 Поз. 8 Поз. 9 
  A:      1      8      1      0      0      2      7      2      1 
  C:      6      2      1      0      0      6      0      0      8 
  G:      1      0      8     10      0      0      3      8      0 
  T:      2      0      0      0     10      2      0      0      1 

2. PPM с псевдосчётом α=0.1 (округлено до 3 знаков):
      Поз. 1 Поз. 2 Поз. 3 Поз. 4 Поз. 5 Поз. 6 Поз. 7 Поз. 8 Поз. 9 
  A:  0.106  0.779  0.106  0.010  0.010  0.202  0.683  0.202  0.106 
  C:  0.587  0.202  0.106  0.010  0.010  0.587  0.010  0.010  0.779 
  G:  0.106  0.010  0.779  0.971  0.010  0.010  0.298  0.779  0.010 
  T:  0.202  0.010  0.010  0.010  0.971  0.202  0.010  0.010  0.106 

   Проверка сумм по столбцам:
     Поз.1: 1.000000
     Поз.2: 1.000000
     Поз.3: 1.000000
     Поз.4: 1.000000
     Поз.5: 1.000000
     Поз.6: 1.000000
     Поз.7: 1.000000
     Поз.8: 1.000000
     Поз.9: 1.0000